# 01 · Import MIMIC-IV-on-FHIR demo → WintEHR
Load the [MIMIC-IV Clinical Database Demo on FHIR (v2.1.0)](https://physionet.org/content/mimic-iv-fhir-demo/2.1.0/)
— 100 de-identified patients as ready-made FHIR R4 NDJSON — into a WintEHR HAPI server as
idempotent `PUT`-by-id transaction Bundles.

**What it handles for you:** dependency ordering (Organization → … → Observations, so HAPI's
referential integrity never trips), patient subsetting (the full demo is ~929k resources;
default imports the 5 richest patients), a provenance `meta.tag` on every imported resource
(so verification and cleanup can find exactly what this notebook wrote), and structural
validation before anything is sent.

**Resource types:** Patient, Encounter (hosp/ED/ICU), Condition, Procedure, Medication +
Request/Dispense/Administration/Statement, Specimen, Observation (labs, micro, ED vitals,
ICU flowsheet), Location, Organization.

_MIMIC dates are de-identified (shifted ~a century forward) but internally consistent —
expect encounter years like 2180._

In [ ]:
import json, gzip, time, pathlib
from collections import Counter, defaultdict
import requests

# ---- Config: edit these -------------------------------------------------
DATA_DIR   = pathlib.Path.home() / "Downloads" / "mimic-iv-clinical-database-demo-on-fhir-2.1.0" / "fhir"

# WintEHR FHIR endpoints (pick one):
#   local dev stack          : http://localhost:8888/fhir        (HAPI direct)
#   wintehrdev via SSH tunnel: http://localhost:18888/fhir       (ssh -L 18888:localhost:8888 azureuser@wintehrdev...)
#   wintehrdev via app proxy : http://wintehrdev.eastus.cloudapp.azure.com/fhir
FHIR_BASE  = "http://localhost:8888/fhir"

LIMIT_PATIENTS = 5      # 5 richest patients ≈ a few 10k resources. None = all 100 (~929k, hours)
PATIENT_IDS    = []     # explicit MimicPatient UUIDs override LIMIT_PATIENTS
INCLUDE_ICU_FLOWSHEET = False  # Chartevents/Datetime/Outputevents = 75% of the dataset's volume
BATCH_SIZE     = 200    # resources per transaction Bundle

# Every imported resource gets this tag — verification + cleanup key off it.
TAG = {"system": "http://wintehr.local/fhir/import", "code": "mimic-iv-fhir-demo-2.1.0"}

# ---- Dependency-ordered file list (referential integrity holds at every step)
FILES = [
    "MimicOrganization", "MimicLocation",                      # infrastructure
    "MimicMedication", "MimicMedicationMix",                   # Mix ingredients ref Medication
    "MimicPatient",
    "MimicEncounter", "MimicEncounterED", "MimicEncounterICU", # ICU partOf hosp Encounter
    "MimicSpecimen", "MimicSpecimenLab",                       # Observations ref Specimen
    "MimicMedicationRequest",                                  # Dispense/Admin ref the Request
    "MimicMedicationDispense", "MimicMedicationDispenseED",
    "MimicMedicationAdministration", "MimicMedicationAdministrationICU",
    "MimicMedicationStatementED",
    "MimicCondition", "MimicConditionED",
    "MimicProcedure", "MimicProcedureED", "MimicProcedureICU",
    # The microbiology trio cross-references BOTH ways (Test --hasMember-->
    # Org --hasMember--> Susc, while Org/Susc --derivedFrom--> back up), so no
    # file order satisfies referential integrity. They are merged into ONE
    # group below and ingested in patient-closed transactions.
    ("Microbiology", ["MimicObservationMicroTest", "MimicObservationMicroOrg",
                      "MimicObservationMicroSusc"]),
    "MimicObservationLabevents", "MimicObservationED", "MimicObservationVitalSignsED",
]
ICU_FLOWSHEET = ["MimicObservationChartevents", "MimicObservationDatetimeevents",
                 "MimicObservationOutputevents"]
if INCLUDE_ICU_FLOWSHEET:
    FILES += ICU_FLOWSHEET

# Loaded in full regardless of patient selection (shared, tiny):
SHARED = {"MimicOrganization", "MimicLocation", "MimicMedication", "MimicMedicationMix"}

def read_ndjson(name):
    with gzip.open(DATA_DIR / f"{name}.ndjson.gz", "rt") as f:
        for line in f:
            yield json.loads(line)

print("data dir :", DATA_DIR)
print("target   :", FHIR_BASE)
print("files    :", len(FILES), "| ICU flowsheet:", "included" if INCLUDE_ICU_FLOWSHEET else "skipped")

## 1 · Inventory the dataset
Count resources per file and confirm the server is reachable before doing any work.

In [ ]:
counts = {}
for entry in FILES:
    for name in (entry[1] if isinstance(entry, tuple) else [entry]):
        counts[name] = sum(1 for _ in read_ndjson(name))
        print(f"{name:38s} {counts[name]:8d}")
print(f"{'TOTAL (selected files)':38s} {sum(counts.values()):8d}")

meta = requests.get(f"{FHIR_BASE}/metadata", timeout=30).json()
print("\nserver:", meta.get("software", {}).get("name"), meta.get("software", {}).get("version"),
      "| FHIR", meta.get("fhirVersion"))

## 2 · Select patients
The demo holds 100 patients but resource volume is heavily skewed. Rank patients by how much
clinical data they carry (encounters + meds + labs) and keep the top `LIMIT_PATIENTS` —
or set `PATIENT_IDS` explicitly. Shared resources (Organization/Location/Medication) always load.

In [ ]:
patients = list(read_ndjson("MimicPatient"))
by_id = {p["id"]: p for p in patients}

# Rank by data richness over the moderate-sized files (fast full scans)
richness = Counter()
for name in ["MimicEncounter", "MimicMedicationRequest", "MimicObservationLabevents", "MimicCondition"]:
    for r in read_ndjson(name):
        ref = (r.get("subject") or {}).get("reference", "")
        if ref.startswith("Patient/"):
            richness[ref.split("/", 1)[1]] += 1

if PATIENT_IDS:
    selected = [pid for pid in PATIENT_IDS if pid in by_id]
elif LIMIT_PATIENTS:
    selected = [pid for pid, _ in richness.most_common(LIMIT_PATIENTS)]
else:
    selected = list(by_id)

sel = set(selected)
print(f"selected {len(sel)} / {len(patients)} patients:")
for pid in selected:
    p = by_id[pid]
    print(f"  {pid}  {p['name'][0]['family']:18s} {p.get('gender','?'):7s} b.{p.get('birthDate','?')}"
          f"  ~{richness[pid]} core resources")

## 3 · Load + filter (dependency order)
Keep a resource if it is shared infrastructure or belongs to a selected patient
(any `Patient/…` reference it carries). Each kept resource gets the provenance tag.

In [ ]:
def patient_refs(o, out):
    if isinstance(o, dict):
        for k, v in o.items():
            if k == "reference" and isinstance(v, str) and v.startswith("Patient/"):
                out.add(v.split("/", 1)[1])
            else:
                patient_refs(v, out)
    elif isinstance(o, list):
        for v in o:
            patient_refs(v, out)
    return out

def tag(r):
    r.setdefault("meta", {}).setdefault("tag", []).append(dict(TAG))
    return r

batches = []   # (group_name, [resources], patient_closed) in dependency order
kept = 0
for entry in FILES:
    name, parts = entry if isinstance(entry, tuple) else (entry, [entry])
    patient_closed = isinstance(entry, tuple)   # merged groups chunk on patient boundaries
    rows = []
    for part in parts:
        for r in read_ndjson(part):
            if part in SHARED:
                rows.append(tag(r))
            elif part == "MimicPatient":
                if r["id"] in sel:
                    rows.append(tag(r))
            else:
                refs = patient_refs(r, set())
                if refs and refs <= sel:
                    r["_pt"] = next(iter(refs))   # stripped before send
                    rows.append(tag(r))
    if patient_closed:
        rows.sort(key=lambda r: r.get("_pt", ""))  # group each patient contiguously
    batches.append((name, rows, patient_closed))
    total = sum(counts[p] for p in parts)
    kept += len(rows)
    print(f"{name:38s} kept {len(rows):7d} / {total:7d}")
print(f"{'TOTAL to import':38s} {kept:12d}")

## 4 · Structural validation
No external deps: unique type/id pairs, and every internal reference to a type we import
must resolve within the import set (references we don't import at all are listed, not fatal).

In [ ]:
def all_refs(o, out):
    if isinstance(o, dict):
        for k, v in o.items():
            if k == "reference" and isinstance(v, str) and "/" in v:
                out.append(v)
            else:
                all_refs(v, out)
    elif isinstance(o, list):
        for v in o:
            all_refs(v, out)
    return out

ids, dupes = set(), 0
imported_types = set()
for name, rows, _pc in batches:
    for r in rows:
        key = f"{r['resourceType']}/{r['id']}"
        dupes += key in ids
        ids.add(key)
        imported_types.add(r["resourceType"])

unresolved = Counter()
for name, rows, _pc in batches:
    for r in rows:
        for ref in all_refs(r, []):
            rtype = ref.split("/", 1)[0]
            if rtype in imported_types and ref not in ids:
                unresolved[rtype] += 1

print(f"resources   : {len(ids)}   duplicates: {dupes}")
print(f"unresolved  : {sum(unresolved.values())}   {dict(unresolved) if unresolved else ''}")
print("OK" if dupes == 0 and not unresolved else "CHECK FAILURES ABOVE")

## 5 · Ingest — transaction Bundles, PUT-by-id
`PUT ResourceType/id` in a `transaction` Bundle = idempotent upsert: re-running the notebook
overwrites in place, never duplicates. Files go up in dependency order so HAPI's referential
integrity is satisfied at every step.

In [ ]:
def put_bundle(rows):
    bundle = {"resourceType": "Bundle", "type": "transaction", "entry": [
        {"resource": r, "request": {"method": "PUT", "url": f"{r['resourceType']}/{r['id']}"}}
        for r in rows]}
    # Trailing slash matters: WintEHR's backend /fhir proxy 301s a bare POST
    # /fhir (and the redirect turns into a GET); HAPI direct accepts both.
    resp = requests.post(FHIR_BASE.rstrip("/") + "/", json=bundle,
                         headers={"Content-Type": "application/fhir+json"}, timeout=300)
    if resp.status_code not in (200, 201):
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:800]}")
    return resp.json()

def chunks_of(rows, patient_closed):
    # <= BATCH_SIZE per bundle; patient-closed groups never split a patient
    # (their cross-references must resolve inside one transaction).
    if not patient_closed:
        for i in range(0, len(rows), BATCH_SIZE):
            yield rows[i:i + BATCH_SIZE]
        return
    chunk, cur = [], None
    for r in rows:
        if len(chunk) >= BATCH_SIZE and r.get("_pt") != cur:
            yield chunk; chunk = []
        cur = r.get("_pt")
        chunk.append(r)
    if chunk:
        yield chunk

t0, sent = time.time(), 0
for name, rows, patient_closed in batches:
    if not rows:
        continue
    for chunk in chunks_of(rows, patient_closed):
        for r in chunk:
            r.pop("_pt", None)   # internal bookkeeping, not FHIR
        put_bundle(chunk)
        sent += len(chunk)
    print(f"{name:38s} {len(rows):7d} sent   ({sent}/{kept}, {time.time()-t0:5.0f}s)")
print(f"\ndone: {sent} resources in {time.time()-t0:.0f}s")

## 6 · Verify
Count what the server now holds under this import's tag, and spot-check one patient's record.

In [ ]:
tag_param = f"{TAG['system']}|{TAG['code']}"
print(f"{'type':24s} {'imported':>9s} {'on server':>10s}")
mismatch = False
for rtype in sorted(imported_types):
    local = sum(1 for _ in ids if _.startswith(rtype + "/"))
    r = requests.get(f"{FHIR_BASE}/{rtype}", params={"_tag": tag_param, "_summary": "count"}, timeout=60)
    server = r.json().get("total", "?")
    flag = "" if server == local else "  <-- MISMATCH"
    mismatch |= bool(flag)
    print(f"{rtype:24s} {local:9d} {server:>10} {flag}")

pid = selected[0]
ev = requests.get(f"{FHIR_BASE}/Patient/{pid}/$everything", params={"_summary": "count"}, timeout=120).json()
print(f"\nPatient/{pid} $everything -> {ev.get('total')} resources")
print("open in WintEHR: <app base URL>/patients — search family name:", by_id[pid]["name"][0]["family"])
print("\nALL COUNTS MATCH" if not mismatch else "SOME COUNTS MISMATCH — see above")

## 7 · (Optional) remove this import
Deletes exactly the resources this notebook tagged, in **reverse** dependency order (children
before the Patients/Medications they reference). Guarded — flip `DO_CLEANUP` deliberately.

In [ ]:
DO_CLEANUP = False

if DO_CLEANUP:
    deleted = 0
    for name, rows, _pc in reversed(batches):
        for i in range(0, len(rows), BATCH_SIZE):
            chunk = rows[i:i + BATCH_SIZE]
            bundle = {"resourceType": "Bundle", "type": "transaction", "entry": [
                {"request": {"method": "DELETE", "url": f"{r['resourceType']}/{r['id']}"}}
                for r in chunk]}
            requests.post(FHIR_BASE.rstrip("/") + "/", json=bundle,
                          headers={"Content-Type": "application/fhir+json"}, timeout=300).raise_for_status()
            deleted += len(chunk)
        if rows:
            print(f"{name:38s} deleted {len(rows)}")
    print("total deleted:", deleted)
else:
    print("Cleanup disabled. Set DO_CLEANUP = True and re-run this cell to remove the import.")